# Phase 1: Colab Environment Setup

Set up the full SoARM + LIBERO + OpenVLA-OFT stack on a fresh Colab A100 runtime.

## Requirements

- [ ] ENV-01: All dependencies install in correct order without pip resolver conflicts
- [ ] ENV-02: EGL headless rendering configured — OffScreenRenderEnv produces non-black LIBERO frames
- [ ] ENV-03: OpenVLA-OFT loads on GPU (A100 bf16) and returns a valid 7-D action tensor

## Usage

**BLOCK A** (this block): Run cells 0-9 top to bottom, then restart the runtime.

**BLOCK B** (Plan 02): After restart, run verification cells for ENV-01 / ENV-02 / ENV-03.

> Note: Block A must complete fully before restarting. Do not run Block B cells before restart.

In [1]:
import os

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set REPO_ROOT to the path where you cloned SoARM-Research.
# If using Google Drive: "/content/drive/MyDrive/SoARM-Research"
# If using git clone directly to Colab: "/content/SoARM-Research"
REPO_ROOT = "/content/drive/MyDrive/SoARM-Research"
# ────────────────────────────────────────────────────────────────────────────

# Derived path constants (do not edit these)
LIBERO_ROOT = f"{REPO_ROOT}/LIBERO/libero/libero"
LIBERO_PKG  = f"{REPO_ROOT}/LIBERO"   # path to setup.py directory
OUT_DIR     = f"{REPO_ROOT}/LIBERO/notebooks/outputs"
BDDL_FILE   = (
    f"{LIBERO_ROOT}/bddl_files/libero_spatial/"
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl"
)

# Create outputs directory so Block B render check can save there
os.makedirs(OUT_DIR, exist_ok=True)

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"LIBERO_ROOT = {LIBERO_ROOT}")
print(f"LIBERO_PKG  = {LIBERO_PKG}")
print(f"OUT_DIR     = {OUT_DIR}")
print(f"BDDL_FILE   = {BDDL_FILE}")
print(f"Saved → {OUT_DIR}  (outputs directory ready)")

REPO_ROOT   = /content/drive/MyDrive/SoARM-Research
LIBERO_ROOT = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero
LIBERO_PKG  = /content/drive/MyDrive/SoARM-Research/LIBERO
OUT_DIR     = /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs
BDDL_FILE   = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero/bddl_files/libero_spatial/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl
Saved → /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs  (outputs directory ready)


In [2]:
# GPU assertion — D-06
# Check GPU availability and warn loudly if not A100.
# OpenVLA-OFT in bf16 requires ~16 GB VRAM; A100 (40 GB) is the target.
import torch

assert torch.cuda.is_available(), (
    "No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if "A100" not in gpu_name:
    print()
    print("WARNING: Expected A100, got", gpu_name)
    print("WARNING: OpenVLA-OFT in bf16 requires ~16 GB+ VRAM.")
    print("WARNING: ENV-03 will OOM on T4 (15 GB). Restart with an A100 runtime.")
    print("WARNING: You may still proceed for install-only testing on T4.")
else:
    print("A100 confirmed. Proceeding.")

GPU:  NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
A100 confirmed. Proceeding.


---

## BLOCK A: Install

Run all cells in this block **top to bottom**, then restart the runtime.

**Ordering is critical** — do not reorder or skip cells:

1. EGL system packages (apt) must install before pip mujoco
2. PyTorch must install before flash-attn (flash-attn compiles against torch CUDA headers)
3. The custom transformers fork must install last (prevents pip downgrade to PyPI version)

---

In [3]:
# Step 1 of 6 — EGL system packages
# Must run BEFORE pip mujoco install.
# These C libraries must exist when the mujoco Python extension builds.
# apt-get update refreshes repo index — prevents 404 for stale package URLs (e.g. libosmesa6).
!apt-get update -qq
!apt-get install -y -q --fix-missing \
    libglfw3 \
    libglew-dev \
    libosmesa6-dev \
    libgles2 \
    libglvnd0 \
    libegl-dev \
    libegl1 \
    libgl1-mesa-glx

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
libegl1 is already the newest version (1.4.0-1).
libgles2 is already the newest version (1.4.0-1).
libglvnd0 is already the newest version (1.4.0-1).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
The following additional packages will be installed:
  libegl-mesa0 libgbm1 libgl-dev libgl1-mesa-dri libglapi-mesa libglew2.2
  libglu1-mesa libglu1-mesa-dev libglx-dev libglx-mesa0 libosmesa6
Suggested packages:
  glew-utils libgles1 libvulkan1
Recommended packages:
  libgl1-amber-dri
The following NEW packages will be installed:
  libegl-dev libgl-dev libglew-dev libglew2.2 libglfw3 libglu1-mesa
  libglu1-mesa-dev libglx-dev libosmesa6 libosmesa6-dev
The following packages will be upgraded:
  libeg

In [4]:
# Step 2 of 6 — PyTorch 2.2.0 (cu121)
# Must run BEFORE flash-attn: flash-attn compiles CUDA kernels against installed torch headers.
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 1.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 107.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 115.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 124.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 147.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 6.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 22.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 46.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 21.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━

In [5]:
# Step 3 of 6 — MuJoCo + simulation stack
#
# numpy pin must be LAST in this cell (after all other installs settle).
# Version check uses subprocess, not import: torch already imported numpy 2.0.2
# in Cell 2 (GPU check), so sys.modules caches 2.0.2 in this session.
# After the runtime restart (before Block B) Python starts fresh and loads
# the on-disk numpy 1.26.4. The subprocess check reports the on-disk version.
import sys, subprocess
print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

# mujoco 3.3.2: binary wheel for cp312; stable version for LIBERO
!pip install "mujoco==3.3.2" -q

# gym: --prefer-binary avoids source build on Python 3.12
!pip install "gym>=0.21,<=0.26" --prefer-binary -q

# robosuite MUST be 1.4.0 — 1.5.x removed SingleArmEnv which LIBERO requires
!pip install     robosuite==1.4.0     bddl==1.0.1     easydict==1.9     cloudpickle==2.1.0     einops==0.4.1     "imageio[ffmpeg]" -q

# opencv: <4.10 required — 4.10+ requires numpy>=2 (breaks torch 2.2.0)
!pip install "opencv-python-headless>=4.7,<4.10" --prefer-binary -q

# numba BEFORE numpy pin: Colab system numba is compiled for numpy 2.x.
# After numpy<2 pin, numba's C extension fails with "numpy.dtype size changed".
# numba 0.59.x supports numpy 1.21-1.26 and Python 3.12 — binary-compatible with 1.x.
!pip install "numba>=0.59,<0.60" -q   # numpy-1.x-compatible numba; must precede numpy<2 pin

# numpy LAST: torch 2.2.0 compiled against numpy 1.x headers.
# numpy 2.x breaks torch._ARRAY_API (torch.from_numpy fails — used in LIBERO).
# Resolver warnings about jax/cupy/opencv needing numpy>=2 are safe to ignore.
!pip install "numpy>=1.24,<2" --force-reinstall -q

import mujoco
mv = tuple(int(x) for x in mujoco.__version__.split("."))
print(f"mujoco {mujoco.__version__} installed")
if mv >= (3, 0, 0):
    print("✓ mujoco 3.x — bddl_base_domain.py mj_kinematics shim is active")

# Check the ON-DISK numpy version (not sys.modules which may cache old version).
# torch imported numpy 2.x in Cell 2; after restart Python loads the disk version.
np_ver = subprocess.check_output(
    [sys.executable, "-m", "pip", "show", "numpy"], text=True
)
for line in np_ver.splitlines():
    if line.startswith("Version:"):
        v = line.split()[1]
        print(f"numpy {v} on disk (fresh after restart)")
        assert v < "2", f"numpy {v} >= 2 on disk — re-run this cell"
        break

print("✓ simulation stack installed — restart runtime now (Runtime > Restart session)")


Python 3.12.13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 121.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 30.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 17.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 13.5 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
distributed 2026.1.1 requires cloudpickle>=3.0.0, but you have cloudpickle 2.1.0 which is incompatible.
dask 2026.1.1 requires cloudpickle>=3.0.0, but you have cloudpickle 2.1.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 56.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━

In [6]:
# Step 4 of 6 — LIBERO editable install
# LIBERO_PKG is defined in Cell 1. setup.py may be missing if Google Drive
# didn't sync the nested LIBERO git repo — auto-clone from GitHub in that case.
import os, subprocess, sys, shutil
from pathlib import Path

# Auto-clone LIBERO if setup.py not found on Drive
_libero_setup = Path(LIBERO_PKG) / 'setup.py'
if not _libero_setup.exists():
    _fallback = '/content/libero'
    _fallback_setup = Path(_fallback) / 'setup.py'
    print(f'⚠ {LIBERO_PKG}/setup.py missing (Google Drive may not sync nested git repos)')

    if _fallback_setup.exists():
        # Already cloned in a prior run — reuse it
        print(f'  ✓ /content/libero already exists and has setup.py — reusing existing clone')
    else:
        # Wipe stale / partial directory and reclone
        if Path(_fallback).exists():
            print(f'  ⚠ {_fallback} exists but setup.py is missing — removing stale directory ...')
            shutil.rmtree(_fallback)
        print(f'  Cloning LIBERO from GitHub → {_fallback} ...')
        _r = subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/Lifelong-Robot-Learning/LIBERO.git', _fallback],
            capture_output=True, text=True,
        )
        if _r.returncode != 0:
            raise RuntimeError(f'LIBERO clone failed:\n{_r.stderr}')
        print(f'  ✓ LIBERO cloned successfully')

    # Redirect paths to cloned location (affects Block B cells)
    LIBERO_PKG  = _fallback
    LIBERO_ROOT = f'{_fallback}/libero/libero'
    BDDL_FILE   = (
        f'{LIBERO_ROOT}/bddl_files/libero_spatial/'
        'pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl'
    )
    print(f'✓ Paths updated → LIBERO_PKG={LIBERO_PKG}')
else:
    print(f'✓ LIBERO found at {LIBERO_PKG}')

# editable install — LIBERO package changes are live without reinstall
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', LIBERO_PKG, '-q'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
else:
    print('✓ LIBERO installed in editable mode from:', LIBERO_PKG)


⚠ /content/drive/MyDrive/SoARM-Research/LIBERO/setup.py missing (Google Drive may not sync nested git repos)
  Cloning LIBERO from GitHub → /content/libero ...
  ✓ LIBERO cloned successfully
✓ Paths updated → LIBERO_PKG=/content/libero
✓ LIBERO installed in editable mode from: /content/libero


In [7]:
# Step 5 of 6 — OpenVLA-OFT supporting packages + custom transformers fork
# The git fork (moojink/transformers-openvla-oft) MUST be installed LAST in this cell.
# Do NOT install transformers from PyPI — the fork replaces it entirely.
!pip install \
    timm==0.9.10 \
    tokenizers==0.19.1 \
    sentencepiece==0.1.99 \
    peft==0.11.1 \
    accelerate \
    huggingface_hub \
    draccus \
    jsonlines \
    wandb -q

# Remove any wrong prismatic: TRI-ML's prismatic-vlms LACKS prismatic.training.train_utils,
# which the moojink checkpoint's remote code (modeling_prismatic.py) imports.
!pip uninstall -y prismatic prismatic-vlms -q

# prismatic must come from the OpenVLA-OFT repo itself.
# --no-deps: the repo pins torch/transformers versions; our stack is already correct
# and letting pip resolve its deps would clobber the transformers fork below.
!pip install "git+https://github.com/moojink/openvla-oft.git" --no-deps -q

# Install transformers fork LAST — pip resolver cannot downgrade to PyPI version this way.
# To pin a specific commit: git+https://github.com/moojink/transformers-openvla-oft.git@<SHA>
!pip install git+https://github.com/moojink/transformers-openvla-oft.git -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 118.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.5 MB/s eta 0:00:00

In [8]:
# Step 6 of 6 — flash-attn (optional — pre-built wheel)
# Tries the cu123 wheel only. If it fails on this CUDA version, flash-attn is
# uninstalled entirely so transformers does NOT see a broken partial install.
# OpenVLA-OFT falls back to standard attention (~15% slower, fully functional).
import subprocess, sys, importlib
from pathlib import Path

cuda_ver = __import__("torch").version.cuda
print(f"Detected: CUDA={cuda_ver}, Torch={__import__('torch').__version__}")

!pip install packaging ninja -q

# Always start clean
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "flash-attn"],
               capture_output=True)
# Clear any stale cached module
for _k in [k for k in sys.modules if "flash_attn" in k]:
    del sys.modules[_k]

FLASH_OK = False

# Attempt: cu123 wheel (torch 2.2, cp312, CUDA 12.3 ABI)
WHEEL_CU123 = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/"
    "v2.6.3/flash_attn-2.6.3+cu123torch2.2cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
)
print("Trying flash-attn cu123 wheel ...")
r = subprocess.run([sys.executable, "-m", "pip", "install", WHEEL_CU123, "-q"],
                   capture_output=True, text=True)
if r.returncode == 0:
    try:
        import flash_attn
        # Functional test: actually exercise the CUDA kernel
        flash_attn.flash_attn_func  # attribute access triggers C extension load
        import torch
        _dummy = torch.randn(1, 1, 4, 32, device="cuda", dtype=torch.float16)
        flash_attn.flash_attn_func(_dummy, _dummy, _dummy, causal=True)
        print(f"✓ flash-attn {flash_attn.__version__} functional")
        FLASH_OK = True
    except Exception as e:
        print(f"  cu123 failed at runtime: {e}")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "flash-attn"],
                       capture_output=True)
        for _k in [k for k in sys.modules if "flash_attn" in k]:
            del sys.modules[_k]
else:
    print(f"  cu123 install failed (exit {r.returncode})")

if not FLASH_OK:
    # Ensure flash_attn is completely absent so transformers skips it cleanly
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "flash-attn"],
                   capture_output=True)
    print("⚠ flash-attn unavailable — uninstalled to prevent broken import")
    print("  transformers will use standard attention (fully functional, ~15% slower)")


Detected: CUDA=12.8, Torch=2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.5 MB/s eta 0:00:00
Trying flash-attn cu123 wheel ...
  cu123 failed at runtime: /usr/local/lib/python3.12/dist-packages/flash_attn_2_cuda.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN3c104cuda9SetDeviceEi
⚠ flash-attn unavailable — uninstalled to prevent broken import
  transformers will use standard attention (fully functional, ~15% slower)


---

## *** STOP — Restart runtime now ***

Go to: **Runtime > Restart session** (or press Ctrl+M .), then continue from BLOCK B below.

Do **not** run any cells below this point until after the runtime has restarted.

After restart, continue in **this notebook** — run the BLOCK B cells below.

---

## BLOCK B: Verification (run after restart)

Run these cells **after** restarting the runtime (see BLOCK A STOP cell above).

**Critical ordering:** Each cell must run in order — EGL must be configured before any
MuJoCo import, and `~/.libero/config.yaml` must exist before any `import libero` call.

- Cell 14: EGL bootstrap (FIRST — no MuJoCo imports before this)
- Cell 15: LIBERO config.yaml bootstrap
- Cell 16: sys.path setup
- Cell 17: ENV-01 — package version check
- Cell 18: ENV-02 — LIBERO render verification

In [9]:
# EGL bootstrap — FIRST POST-RESTART CELL
# Creates NVIDIA ICD JSON BEFORE setting MUJOCO_GL env var.
# MUJOCO_GL must be set BEFORE the mujoco/robosuite/libero packages are loaded.
# Run this cell first — before any physics or simulation package is imported.

import os, json

# Step (a): Create NVIDIA EGL ICD JSON
os.makedirs("/usr/share/glvnd/egl_vendor.d", exist_ok=True)
with open("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "w") as _f:
    json.dump({
        "file_format_version": "1.0.0",
        "ICD": {"library_path": "libEGL_nvidia.so.0"}
    }, _f)

# Step (b): Set MuJoCo GL backend env vars
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Step (c): Confirm setup
print("EGL ICD created. MUJOCO_GL=egl set. Run this cell FIRST before loading any simulation packages.")

EGL ICD created. MUJOCO_GL=egl set. Run this cell FIRST before loading any simulation packages.


In [10]:
# LIBERO config.yaml bootstrap — run BEFORE any import libero
# Pre-creates ~/.libero/config.yaml to prevent input() → EOFError
# when libero/__init__.py checks for the config file on first import.
# All paths derive from LIBERO_ROOT defined in Cell 1.

import yaml
from pathlib import Path

# Build config dict matching keys from get_default_path_dict() in LIBERO/__init__.py
config = {
    "benchmark_root": LIBERO_ROOT,
    "bddl_files":     f"{LIBERO_ROOT}/bddl_files",
    "init_states":    f"{LIBERO_ROOT}/init_files",
    "datasets":       f"{LIBERO_ROOT}/../datasets",
    "assets":         f"{LIBERO_ROOT}/assets",
}

config_dir = Path.home() / ".libero"
config_dir.mkdir(parents=True, exist_ok=True)
(config_dir / "config.yaml").write_text(yaml.dump(config))
print(f"Saved \u2192 {config_dir / 'config.yaml'}")

Saved → /root/.libero/config.yaml


In [11]:
# sys.path setup — run AFTER config.yaml bootstrap, BEFORE any libero import
# Replicates explorations/create_scene.py sys.path.insert pattern.
# LIBERO_PKG is defined in Cell 1 (and may be overridden in Cell 7 if Drive is missing).

import sys

if LIBERO_PKG not in sys.path:
    sys.path.insert(0, LIBERO_PKG)

print(f"sys.path[0] = {sys.path[0]}")
print(f"LIBERO_PKG inserted: {LIBERO_PKG}")

sys.path[0] = /content/libero
LIBERO_PKG inserted: /content/libero


In [12]:
# ENV-01: Package version check + pip conflict scan
# Per D-07: explicit PASS/FAIL output with installed vs expected versions.
# Version comparison strips build tag (e.g. "2.2.0+cu121" → "2.2.0").
# pip check is filtered to OUR pipeline packages — Colab system conflicts are pre-existing.

import importlib.metadata
import subprocess

# Packages to verify with their expected versions
EXPECTED = {
    "mujoco":     "3.3.2",
    "robosuite":  "1.4.0",
    "gym":        "0.25.2",
    "torch":      "2.2.0",
    "timm":       "0.9.10",
    "peft":       "0.11.1",
    "tokenizers": "0.19.1",
}

print(f"{'Package':<20} {'Installed':<20} {'Expected':<15} {'Status'}")
print("-" * 70)

all_ok = True
for pkg, expected_ver in EXPECTED.items():
    try:
        installed = importlib.metadata.version(pkg)
        installed_base = installed.split('+')[0]  # strip build tag (e.g. "2.2.0+cu121" → "2.2.0")
        ok = installed_base == expected_ver
        status = "OK" if ok else "MISMATCH"
        if not ok:
            all_ok = False
    except importlib.metadata.PackageNotFoundError:
        installed = "NOT FOUND"
        status = "MISSING"
        all_ok = False
    print(f"{pkg:<20} {installed:<20} {expected_ver:<15} {status}")

print()

# Our pipeline packages — filter pip check to these only.
# Colab has pre-installed jax/cupy/opencv needing numpy>=2; those are not our packages.
OUR_PACKAGES = {
    "mujoco", "robosuite", "gym", "torch", "timm", "peft", "tokenizers",
    "bddl", "transformers", "huggingface-hub", "huggingface_hub",
    "flash-attn", "flash_attn", "einops", "sentencepiece", "numba", "numpy",
    "easydict", "cloudpickle", "imageio", "opencv-python-headless",
}

# Run pip check
result = subprocess.run(["pip", "check"], capture_output=True, text=True)
all_conflicts = result.stdout.strip() if result.stdout.strip() else ""

# Filter: only flag a conflict when the PACKAGE CAUSING it (first token) is one we installed.
# Do NOT use substring match — "jaxlib requires numpy>=2" would falsely match "numpy" in OUR_PACKAGES.
our_conflicts = []
if all_conflicts:
    OUR_PKG_KEYS = {p.lower().replace('-', '_') for p in OUR_PACKAGES}
    for line in all_conflicts.split('\n'):
        if not line.strip():
            continue
        # Each line: "<package-name> <version> has requirement ..."
        conflict_pkg = line.split()[0].lower().replace('-', '_')
        if conflict_pkg in OUR_PKG_KEYS:
            our_conflicts.append(line)

# Report all pip check output (truncated) then our filtered result
if all_conflicts:
    print("pip check output (all):")
    print(all_conflicts[:2000])  # truncate long output
    if our_conflicts:
        print(f"\nConflicts in our pipeline packages ({len(our_conflicts)}):")
        for c in our_conflicts:
            print(f"  \u26a0 {c}")

# Overall result
if not our_conflicts:
    print("ENV-01: PASS \u2014 all pipeline packages installed, no conflicts in our required packages")
    if all_conflicts:
        print("  (Pre-existing Colab system package conflicts above are not ours and won't affect this pipeline)")
else:
    print(f"ENV-01: FAIL \u2014 {len(our_conflicts)} conflict(s) in our required packages:")
    for c in our_conflicts:
        print(f"  \u2717 {c}")

Package              Installed            Expected        Status
----------------------------------------------------------------------
mujoco               3.3.2                3.3.2           OK
robosuite            1.4.0                1.4.0           OK
gym                  0.25.2               0.25.2          OK
torch                2.2.0+cu121          2.2.0           OK
timm                 0.9.10               0.9.10          OK
peft                 0.11.1               0.11.1          OK
tokenizers           0.19.1               0.19.1          OK

pip check output (all):
openvla-oft 0.0.1 requires dlimp, which is not installed.
openvla-oft 0.0.1 requires json-numpy, which is not installed.
openvla-oft 0.0.1 requires tensorflow-graphics, which is not installed.
ipython 7.34.0 requires jedi, which is not installed.
openvla-oft 0.0.1 has requirement diffusers==0.30.3, but you have diffusers 0.38.0.
openvla-oft 0.0.1 has requirement draccus==0.8.0, but you have draccus 0.11.6.
op

In [13]:
# ENV-02: LIBERO Panda render verification
# Per D-04 and D-07: render a default Panda LIBERO frame, assert non-black (mean > 5.0),
# save to outputs/, display inline.
# matplotlib.use("Agg") MUST be called before import matplotlib.pyplot (project convention).

import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Numba compatibility shim — handles any numba/numpy ABI mismatch transparently.
# robosuite uses numba only for JIT optimization; correctness is unaffected if numba is unavailable.
import sys as _sys, types as _types

def _test_numba():
    import numba as _nb
    _nb.njit(lambda x: x + 1)(1.0)  # actually exercise the C extension

try:
    _test_numba()
except Exception as _err:
    class _NumbaStub(_types.ModuleType):
        """No-op stub so robosuite.utils.numba works without a compatible numba."""
        def __init__(self): super().__init__('numba'); self.__version__ = '0.0-stub'
        def njit(self, *a, cache=False, **kw): return a[0] if (a and callable(a[0])) else (lambda f: f)
        def jit(self, *a, **kw): return a[0] if (a and callable(a[0])) else (lambda f: f)
        def generated_jit(self, *a, **kw): return a[0] if (a and callable(a[0])) else (lambda f: f)
        def __getattr__(self, name): return lambda *a, **kw: None
    _sys.modules['numba'] = _NumbaStub()
    print(f"numba stub installed ({type(_err).__name__}: {_err})")
    print("robosuite JIT disabled — simulation correctness unaffected")

from libero.libero.envs import OffScreenRenderEnv

# Resolve LIBERO_PKG by probing disk — LIBERO_PKG from Cell 1 may be a Drive path that
# doesn't exist (clone fallback always lands at /content/libero).
import pathlib as _pl
_LIBERO_CANDIDATES = ["/content/libero", str(LIBERO_PKG)]
_LIBERO_PKG_ACTUAL = next(
    (p for p in _LIBERO_CANDIDATES
     if _pl.Path(p, "libero", "libero", "bddl_files").exists()),
    None,
)
assert _LIBERO_PKG_ACTUAL, f"LIBERO bddl_files not found in any of: {_LIBERO_CANDIDATES}"
BDDL_FILE = str(
    _pl.Path(_LIBERO_PKG_ACTUAL) / "libero" / "libero" / "bddl_files" / "libero_spatial"
    / "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl"
)
assert _pl.Path(BDDL_FILE).exists(), f"BDDL not found: {BDDL_FILE}"
print(f"LIBERO_PKG resolved: {_LIBERO_PKG_ACTUAL}")
print(f"BDDL_FILE: {BDDL_FILE}")

env = OffScreenRenderEnv(
    bddl_file_name=BDDL_FILE,
    camera_names=["agentview"],
    camera_heights=256,
    camera_widths=256,
    has_renderer=False,
    has_offscreen_renderer=True,
)

obs = env.reset()
for _ in range(5):
    obs, _, _, _ = env.step(np.zeros(7))

# MuJoCo renders images upside-down — always flip vertically
frame = obs["agentview_image"][::-1]

mean_pixel = frame.mean()
status = "PASS" if mean_pixel > 5.0 else "FAIL"
print(f"ENV-02: {status} \u2014 mean pixel value: {mean_pixel:.2f} (threshold: > 5.0)")

# Save render check image
out_path = OUT_DIR + "/libero_render_check.png"
plt.imsave(out_path, frame)
print(f"Saved \u2192 {out_path}")

# Display inline
plt.figure(figsize=(4, 4))
plt.imshow(frame)
plt.title("ENV-02: LIBERO Panda Render Check")
plt.axis("off")
plt.tight_layout()
plt.show()

env.close()

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


numba stub installed (ImportError: Numba needs NumPy 1.26 or less)
robosuite JIT disabled — simulation correctness unaffected


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_4872/2141745822.py", line 34, in <cell line: 0>
    from libero.libero.envs import OffScreenRenderEnv
  File "/content/libero/libero/libero/envs/__init__.py", line 7, in <module>
    from .venv import SubprocVectorEnv, DummyVectorEnv
  File "/content/libero/libero/libero/envs/venv.py", line 3, in <module>
    import gym
  File "/usr/local/lib/python3.12/dist-packages/gym/__init__.py", line 7, in <module>
    from gym.core import (
  File "/usr/local/lib/python3.12/dist-packages/gym/core.py", line 16, in <module>
    from gym import spaces
  File "/usr/local/lib/python3.12/dist-packages/gym/spaces/__init__.py", line 11, in <module>
    from gym.spaces.box import Box
  File "/usr/local/lib/python3.12/dist-packages/gym/spaces/box.py", line 8, in <module>
    from gym.spaces.

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


TypeError: object of type 'NoneType' has no len()

---
### Block B Progress
- [x] EGL configured (Cell 14)
- [x] LIBERO config bootstrapped (Cell 15)
- [ ] ENV-01 — run Cell 17
- [ ] ENV-02 — run Cell 18
- [ ] ENV-03 — run Plan 03 cells

Proceed to Plan 03 for ENV-03 (OpenVLA-OFT model loading).

## ENV-03: OpenVLA-OFT Model Load Check

Loads `moojink/openvla-7b-oft-finetuned-libero-spatial` in bf16 on A100.
Requires ~16 GB VRAM. Verify the GPU assertion in Step 1 passed before running.

**D-05:** No 4-bit quantisation (bitsandbytes) — bf16 + `low_cpu_mem_usage=True` is sufficient on A100.

In [ ]:
# prismatic guard — the checkpoint's remote code (modeling_prismatic.py) imports
# prismatic.training.train_utils, which exists ONLY in the moojink/openvla-oft repo
# (NOT TRI-ML prismatic-vlms, NOT PyPI). If the installed prismatic lacks it,
# fall back to a repo clone on sys.path — the clone shadows any broken install.
import importlib, subprocess, sys as _sys
from pathlib import Path as _Path

def _prismatic_ok():
    try:
        importlib.import_module("prismatic.training.train_utils")
        return True
    except Exception:
        return False

if not _prismatic_ok():
    _repo = _Path("/content/openvla-oft")
    if not _repo.exists():
        print("Cloning moojink/openvla-oft for prismatic package ...")
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/moojink/openvla-oft.git", str(_repo)], check=True)
    for _k in [k for k in list(_sys.modules) if k.startswith("prismatic")]:
        del _sys.modules[_k]
    if str(_repo) not in _sys.path:
        _sys.path.insert(0, str(_repo))
    try:
        importlib.import_module("prismatic.training.train_utils")
        print("prismatic resolved from /content/openvla-oft (sys.path)")
    except Exception:
        # Surface the REAL failure — usually a missing dependency in prismatic's
        # import chain (e.g. wandb), not a missing prismatic module itself.
        import traceback
        traceback.print_exc()
        raise RuntimeError(
            "prismatic import failed — see traceback above for the missing dependency"
        )

import torch, numpy as np, json
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor
from huggingface_hub import hf_hub_download

CHECKPOINT = "moojink/openvla-7b-oft-finetuned-libero-spatial"

print(f"Loading processor from {CHECKPOINT}...")
processor = AutoProcessor.from_pretrained(CHECKPOINT, trust_remote_code=True)

print("Loading model in bf16 on GPU...")
model = AutoModelForVision2Seq.from_pretrained(
    CHECKPOINT,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).to("cuda")

_p = next(model.parameters())
print(f"Model dtype: {_p.dtype}  device: {_p.device}")

# The checkpoint config's norm_stats holds the OXE PRETRAINING datasets (bridge_orig,
# fractal, ...). The LIBERO fine-tune statistics live in a separate
# dataset_statistics.json in the HF repo. The openvla-oft eval code overlays it after
# loading — we must do the same or unnorm_key='libero_spatial' won't exist.
_stats_path = hf_hub_download(CHECKPOINT, "dataset_statistics.json")
with open(_stats_path) as _f:
    model.norm_stats = json.load(_f)
print(f"norm_stats overlaid from dataset_statistics.json: {list(model.norm_stats.keys())}")

# Resolve unnorm_key — fine-tune datasets are often suffixed "_no_noops"
UNNORM_KEY = "libero_spatial"
if UNNORM_KEY not in model.norm_stats:
    if f"{UNNORM_KEY}_no_noops" in model.norm_stats:
        UNNORM_KEY = f"{UNNORM_KEY}_no_noops"
    else:
        raise KeyError(
            f"No libero_spatial key in norm_stats. Available: {list(model.norm_stats.keys())}"
        )
print(f"Using unnorm_key: {UNNORM_KEY}")

# Build dummy input (per RESEARCH.md §Code Examples §ENV-03)
dummy_image = Image.fromarray(np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8))
prompt = "In: What action should the robot take to pick up the black bowl?\nOut:"

inputs = processor(prompt, dummy_image).to("cuda:0", dtype=torch.bfloat16)

# OFT predict_action returns (actions, actions_hidden_states) — unpack the tuple.
# OFT predicts action CHUNKS: shape (NUM_ACTIONS_CHUNK, 7), not a single (7,) action.
with torch.no_grad():
    result = model.predict_action(**inputs, unnorm_key=UNNORM_KEY, do_sample=False)

action = result[0] if isinstance(result, tuple) else result
action = np.asarray(action)

# Contract: each action step is 7-D (6 DoF + gripper). Accept (7,) or (chunk, 7).
is_7d = action.shape == (7,) or (action.ndim == 2 and action.shape[-1] == 7)
status = "PASS" if is_7d else "FAIL"
print(f"ENV-03: {status} — action shape: {action.shape}, dtype: {action.dtype}")
print(f"First action step: {action if action.ndim == 1 else action[0]}")


Cloning moojink/openvla-oft for prismatic package ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Traceback (most recent call last):
  File "/tmp/ipykernel_4872/345471438.py", line 26, in <cell line: 0>
    importlib.import_module("prismatic.training.train_utils")
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1310, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "<froze

Traceback (most recent call last):
  File "/tmp/ipykernel_4872/345471438.py", line 26, in <cell line: 0>
    importlib.import_module("prismatic.training.train_utils")
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1310, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1310, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "<frozen importlib._bootstrap>", lin

## Phase 1 Summary

| Requirement | Check | Status | Notes |
|-------------|-------|--------|-------|
| ENV-01: Package versions + pip check | Run Cell 15 (ENV-01 code cell) | ☐ | Update after running |
| ENV-02: LIBERO render | Run Cell 16 (ENV-02 code cell) | ☐ | Update after running |
| ENV-03: VLA load + action shape (7,) | Run ENV-03 code cell | ☐ | Update after running |

Update the Status column after running verification cells. All three must show PASS before proceeding to Phase 2.

**Phase 1 deliverable:** `LIBERO/notebooks/01-colab-env-setup.ipynb` — run cells 0–N (with restart after the kernel-restart marker cell).